# Gold province reporting-area validation

Persist D-57 dimension, population-fact, and project-fact evidence before enforcing STOP checks. No runtime result is claimed until this notebook runs in Databricks.


In [ ]:
%sql
USE CATALOG `buildabida-capstone`;
CREATE SCHEMA IF NOT EXISTS `04-validation`;

DECLARE OR REPLACE VARIABLE gold_province_validation_rule STRING
    DEFAULT 'gold-province-reporting-validation-v2';

-- d-57: the newest primary population of each level, from the same accepted table c rows
CREATE OR REPLACE TEMPORARY VIEW gold_province_region_population_compare AS
WITH region_population AS (
    SELECT region_key, population_count
    FROM `03-gold`.fact_region_population
    WHERE is_primary_source AND region_key <> 0
    QUALIFY reference_year = MAX(reference_year) OVER ()
),

area_population AS (
    SELECT population.reporting_area_key, population.population_count, area.region_key
    FROM `03-gold`.fact_province_reporting_area_population AS population
    LEFT JOIN `03-gold`.dim_province_reporting_area AS area
        ON population.reporting_area_key = area.reporting_area_key
    WHERE population.is_primary_source
    QUALIFY population.reference_year = MAX(population.reference_year) OVER ()
),

area_by_region AS (
    SELECT region_key, COALESCE(SUM(population_count), 0) AS area_population
    FROM area_population
    WHERE reporting_area_key <> 0
    GROUP BY region_key
)

SELECT
    region_population.region_key,
    COALESCE(region_population.population_count, 0) AS region_population_count,
    COALESCE(area_by_region.area_population, 0) AS area_population,
    (SELECT COALESCE(SUM(population_count), 0) FROM area_population WHERE reporting_area_key = 0)
        AS unresolved_population,
    (SELECT COALESCE(SUM(population_count), 0) FROM area_population) AS all_area_population
FROM region_population
LEFT JOIN area_by_region
    ON region_population.region_key = area_by_region.region_key;

CREATE OR REPLACE TEMPORARY VIEW gold_province_validation_context AS
WITH project AS (
    SELECT MAX(run_id) AS project_run_id, MAX(source_snapshot_id) AS source_snapshot_id
    FROM `03-gold`.fact_project_snapshot WHERE is_current_snapshot = TRUE
), area AS (
    SELECT MAX(run_id) AS area_run_id FROM `03-gold`.dim_province_reporting_area
), population AS (
    SELECT MAX(run_id) AS population_run_id
    FROM `03-gold`.fact_province_reporting_area_population
)
SELECT SHA2(CONCAT_WS('||', project.project_run_id, area.area_run_id,
        population.population_run_id, gold_province_validation_rule), 256)
        AS validation_run_id,
    project.project_run_id AS pipeline_run_id, project.source_snapshot_id
FROM project CROSS JOIN area CROSS JOIN population;


In [ ]:
%sql
CREATE OR REPLACE TEMPORARY VIEW gold_province_reporting_checks AS
WITH checks AS (
    SELECT 'dim_province_reporting_area' AS table_name, 'Uniqueness' AS attribute,
        'reporting area key is unique' AS check_name,
        COUNT(*) - COUNT(DISTINCT reporting_area_key) AS failed_rows, COUNT(*) AS total_rows,
        'stop' AS action
    FROM `03-gold`.dim_province_reporting_area
    UNION ALL
    SELECT 'dim_province_reporting_area', 'Completeness', 'exactly one Unknown member',
        ABS(COUNT_IF(reporting_area_key = 0 AND reporting_area_type = 'UNKNOWN') - 1),
        COUNT(*), 'stop'
    FROM `03-gold`.dim_province_reporting_area
    UNION ALL
    SELECT 'dim_province_reporting_area', 'Validity', 'allowed reporting area types',
        COUNT_IF(reporting_area_type NOT IN ('PROVINCE', 'INDEPENDENT_CITY', 'NCR', 'UNKNOWN')),
        COUNT(*), 'stop'
    FROM `03-gold`.dim_province_reporting_area
    UNION ALL
    SELECT 'dim_province_reporting_area', 'Referential integrity',
        'geographic reporting areas have a parent region',
        COUNT_IF(reporting_area_key <> 0 AND region_key = 0), COUNT(*), 'stop'
    FROM `03-gold`.dim_province_reporting_area
    UNION ALL
    SELECT 'dim_province_reporting_area_boundary', 'Uniqueness', 'boundary key is unique',
        COUNT(*) - COUNT(DISTINCT boundary_key), COUNT(*), 'stop'
    FROM `03-gold`.dim_province_reporting_area_boundary
    UNION ALL
    SELECT 'fact_province_reporting_area_population', 'Uniqueness',
        'population fact key is unique',
        COUNT(*) - COUNT(DISTINCT reporting_area_population_fact_key), COUNT(*), 'stop'
    FROM `03-gold`.fact_province_reporting_area_population
    UNION ALL
    SELECT 'fact_province_reporting_area_population', 'Uniqueness',
        'population natural grain is unique',
        COUNT(*) - COUNT(DISTINCT NAMED_STRUCT('area', reporting_area_key,
            'year', reference_year, 'source', source_name)), COUNT(*), 'stop'
    FROM `03-gold`.fact_province_reporting_area_population
    UNION ALL
    SELECT 'fact_province_reporting_area_population', 'Referential integrity',
        'population has no orphan reporting area key', COUNT_IF(area.reporting_area_key IS NULL),
        COUNT(*), 'stop'
    FROM `03-gold`.fact_province_reporting_area_population AS population
    LEFT JOIN `03-gold`.dim_province_reporting_area AS area
        ON population.reporting_area_key = area.reporting_area_key
    UNION ALL
    SELECT 'fact_province_reporting_area_population', 'Consistency',
        'Gold population total equals Silver reporting area population',
        CASE WHEN SUM(gold.population_count) <=> (
            SELECT SUM(silver.population_count)
            FROM `02-silver`.silver_province_reporting_area_population AS silver
        ) THEN 0 ELSE 1 END, COUNT(*), 'stop'
    FROM `03-gold`.fact_province_reporting_area_population AS gold
    UNION ALL
    SELECT 'fact_project_snapshot', 'Uniqueness', 'project snapshot key is unique',
        COUNT(*) - COUNT(DISTINCT project_snapshot_key), COUNT(*), 'stop'
    FROM `03-gold`.fact_project_snapshot
    UNION ALL
    SELECT 'fact_project_snapshot', 'Referential integrity',
        'project fact has no orphan reporting area key', COUNT_IF(area.reporting_area_key IS NULL),
        COUNT(*), 'stop'
    FROM `03-gold`.fact_project_snapshot AS project
    LEFT JOIN `03-gold`.dim_province_reporting_area AS area
        ON project.reporting_area_key = area.reporting_area_key
    UNION ALL
    SELECT 'fact_project_snapshot', 'Completeness',
        'one fact row per canonical Silver project snapshot',
        ABS(COUNT(*) - (SELECT COUNT(*) FROM `02-silver`.silver_project)), COUNT(*), 'stop'
    FROM `03-gold`.fact_project_snapshot WHERE is_current_snapshot = TRUE
    UNION ALL
    SELECT 'fact_project_snapshot', 'Consistency',
        'current project count is unchanged after reporting area key',
        ABS(COUNT(*) - COUNT(DISTINCT project_key)), COUNT(*), 'stop'
    FROM `03-gold`.fact_project_snapshot WHERE is_current_snapshot = TRUE
    UNION ALL
    SELECT 'fact_project_snapshot', 'Consistency',
        'reported budget total equals canonical Silver projects',
        CASE WHEN SUM(reported_budget_pesos) <=> (
            SELECT SUM(reported_budget_pesos) FROM `02-silver`.silver_project
        ) THEN 0 ELSE 1 END, COUNT(*), 'stop'
    FROM `03-gold`.fact_project_snapshot WHERE is_current_snapshot = TRUE
    UNION ALL
    SELECT 'fact_project_snapshot', 'Lineage', 'current snapshot and lineage are complete',
        COUNT_IF(run_id IS NULL OR source_snapshot_id IS NULL OR source_load_ts IS NULL
            OR reporting_area_key IS NULL), COUNT(*), 'stop'
    FROM `03-gold`.fact_project_snapshot WHERE is_current_snapshot = TRUE
    UNION ALL
    SELECT 'fact_project_snapshot', 'Consistency', 'one current DPWH snapshot',
        GREATEST(COUNT(DISTINCT source_snapshot_id) - 1, 0), COUNT(*), 'stop'
    FROM `03-gold`.fact_project_snapshot WHERE is_current_snapshot = TRUE
    UNION ALL
    SELECT 'dim_province_reporting_area', 'Uniqueness',
        'PSGC reporting area code is unique per PSGC version',
        COUNT_IF(reporting_area_key <> 0) - COUNT(DISTINCT CASE WHEN reporting_area_key <> 0
            THEN NAMED_STRUCT('code', psgc_reporting_area_code, 'version', psgc_version) END),
        COUNT(*), 'stop'
    FROM `03-gold`.dim_province_reporting_area
    UNION ALL
    SELECT 'dim_province_reporting_area', 'Referential integrity',
        'parent region key exists in dim_region', COUNT_IF(region.region_key IS NULL),
        COUNT(*), 'stop'
    FROM `03-gold`.dim_province_reporting_area AS area
    LEFT JOIN `03-gold`.dim_region AS region
        ON area.region_key = region.region_key
    UNION ALL
    SELECT 'fact_province_reporting_area_population', 'Consistency',
        'reporting-area population total equals region population total',
        CASE WHEN MAX(all_area_population) <=> SUM(region_population_count) THEN 0 ELSE 1 END,
        1, 'stop'
    FROM gold_province_region_population_compare
    UNION ALL
    SELECT 'fact_province_reporting_area_population', 'Consistency',
        'no region has more reporting-area population than its own total',
        COUNT_IF(area_population > region_population_count), COUNT(*), 'stop'
    FROM gold_province_region_population_compare
    UNION ALL
    SELECT 'fact_province_reporting_area_population', 'Consistency',
        'region minus reporting-area population equals unresolved population',
        CASE WHEN SUM(region_population_count - area_population) <=> MAX(unresolved_population)
            THEN 0 ELSE 1 END,
        1, 'stop'
    FROM gold_province_region_population_compare
    UNION ALL
    SELECT 'fact_project_snapshot', 'Consistency',
        'reporting area key 0 holds exactly the unresolved and non-geographic projects',
        COUNT_IF((reporting_area_key = 0)
            <> NOT (reporting_area_match_status <=> 'MATCHED'
                OR reporting_area_match_status <=> 'MATCHED_WITH_CONFLICT')),
        COUNT(*), 'stop'
    FROM `03-gold`.fact_project_snapshot WHERE is_current_snapshot = TRUE
    UNION ALL
    SELECT 'fact_project_snapshot', 'Consistency',
        'project region differs from the parent region of its reporting area',
        COUNT_IF(fact.region_key <> 0 AND fact.reporting_area_key <> 0
            AND NOT (fact.region_key <=> area.region_key)),
        COUNT(*), 'flag'
    FROM `03-gold`.fact_project_snapshot AS fact
    LEFT JOIN `03-gold`.dim_province_reporting_area AS area
        ON fact.reporting_area_key = area.reporting_area_key
    WHERE fact.is_current_snapshot = TRUE
    UNION ALL
    SELECT 'fact_project_snapshot', 'Completeness',
        'projects with an official region but no reporting area',
        COUNT_IF(region_key <> 0 AND reporting_area_key = 0), COUNT(*), 'flag'
    FROM `03-gold`.fact_project_snapshot WHERE is_current_snapshot = TRUE
)
SELECT context.validation_run_id, context.pipeline_run_id, 'gold' AS layer,
    checks.table_name, context.source_snapshot_id, gold_province_validation_rule AS rule_version,
    checks.attribute AS data_quality_attribute, checks.check_name AS data_quality_check,
    CAST(checks.failed_rows AS BIGINT) AS failed_rows, CAST(checks.total_rows AS BIGINT) AS total_rows,
    ROUND(100.0 * checks.failed_rows / NULLIF(checks.total_rows, 0), 4) AS percentage,
    CASE
        WHEN checks.action = 'stop' AND checks.failed_rows > 0 THEN 'FAIL'
        WHEN checks.action = 'flag' AND checks.failed_rows > 0 THEN 'FLAG'
        ELSE 'PASS'
    END AS status, checks.action,
    CONCAT('failed=', checks.failed_rows, '; total=', checks.total_rows) AS details,
    CURRENT_TIMESTAMP() AS run_ts
FROM checks CROSS JOIN gold_province_validation_context AS context;


In [ ]:
%sql
CREATE TABLE IF NOT EXISTS `04-validation`.gold_dq_results (
    validation_run_id STRING NOT NULL, pipeline_run_id STRING NOT NULL, layer STRING NOT NULL,
    table_name STRING NOT NULL, source_snapshot_id STRING, rule_version STRING,
    data_quality_attribute STRING NOT NULL, data_quality_check STRING NOT NULL,
    failed_rows BIGINT, total_rows BIGINT, percentage DOUBLE, status STRING NOT NULL,
    action STRING NOT NULL, details STRING, run_ts TIMESTAMP NOT NULL
) USING DELTA COMMENT 'Gold validation evidence. One row per check per validation run.';

MERGE INTO `04-validation`.gold_dq_results AS target
USING gold_province_reporting_checks AS source
ON target.validation_run_id = source.validation_run_id
AND target.table_name = source.table_name
AND target.data_quality_check = source.data_quality_check
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

SELECT ASSERT_TRUE(
    COUNT(*) > 0 AND COUNT_IF(status = 'FAIL' AND action = 'stop') = 0,
    CONCAT(COUNT_IF(status = 'FAIL' AND action = 'stop'),
        ' blocking Gold province-reporting checks failed')
)
FROM `04-validation`.gold_dq_results
WHERE validation_run_id = (SELECT validation_run_id FROM gold_province_validation_context);


# Summary

This validator proves dimension and fact uniqueness, one Unknown, parent and key integrity, unchanged current project grain, unchanged budget, population reconciliation, and complete current lineage. Evidence is persisted before the STOP gate.
